# index-earnings-nowcast — Colab bootstrap

Runs the CLI top to bottom. This notebook holds no business logic: every step is a
shell call into `python -m nowcast.cli`.

Before running: add `OPENROUTER_API_KEY` and `SEC_USER_AGENT` as Colab secrets
(key icon in the left sidebar). Only `:free` OpenRouter models are used.

In [ ]:
REPO_URL = 'https://github.com/your-org/index-earnings-nowcast.git'
BRANCH = 'main'
USE_DRIVE = True            # persist DATA_DIR on Drive across sessions
DATA_SUBDIR = 'nowcast-data'
EXTRACT_LIMIT = 10          # documents to extract in this session

## 1. Clone and install

In [ ]:
%cd /content
!rm -rf index-earnings-nowcast
!git clone --branch {BRANCH} --depth 1 {REPO_URL} index-earnings-nowcast
%cd index-earnings-nowcast
!python -m pip install --quiet --upgrade pip
!python -m pip install --quiet -e '.[dev]'
!test -f requirements.lock.txt && python -m pip install --quiet -r requirements.lock.txt || echo 'no requirements.lock.txt, using pyproject pinning'
print('install complete')

## 2. Persist data on Drive (optional)
`DATA_DIR` is read by the CLI in preference to `paths.data_dir`, so caches, raw
documents and extractions survive a runtime restart.

In [ ]:
import os
from pathlib import Path

data_dir = Path('/content/data')
if USE_DRIVE:
    from google.colab import drive

    drive.mount('/content/drive')
    data_dir = Path('/content/drive/MyDrive') / DATA_SUBDIR
data_dir.mkdir(parents=True, exist_ok=True)
os.environ['DATA_DIR'] = str(data_dir)
print(f'DATA_DIR={data_dir}')

## 3. Secrets
Copied from Colab userdata into the environment. Values are never printed;
only whether each one was found.

In [ ]:
import contextlib
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None

for name in ('OPENROUTER_API_KEY', 'SEC_USER_AGENT'):
    if userdata is not None:
        with contextlib.suppress(Exception):
            os.environ.setdefault(name, userdata.get(name))
    print(f'{name}: ' + ('set' if os.environ.get(name) else 'MISSING'))


## 4. Validate the config

In [ ]:
!python -m nowcast.cli validate-config --strict

## 5. Build the universe (S&P 500 membership + CIKs)
Idempotent: a second run with unchanged inputs makes no HTTP requests.

In [ ]:
!python -m nowcast.cli build-constituents

## 6. Estimate the scope before spending anything
Offline. Prints `ESTIMATE ONLY`; it is not a promise of document count or cost.

In [ ]:
!python -m nowcast.cli estimate-scope --set data.constituents_csv=data/universe/constituents.csv

## 7. Ingest (capped at `data.edgar_max_documents`, default 100)

In [ ]:
!python -m nowcast.cli ingest --set data.constituents_csv=data/universe/constituents.csv

## 8. Extract a small sample with the configured free model

In [ ]:
!python -m nowcast.cli extract --limit {EXTRACT_LIMIT}

## 9. What came out
Read the manifests to see exactly what produced each output.

In [ ]:
import json
from pathlib import Path

for path in sorted(Path(os.environ['DATA_DIR'], 'manifests').glob('*.json')):
    body = json.loads(path.read_text())
    print(f"{path.name}: stage={body.get('stage')} models={body.get('models', 'n/a')} notes={body.get('notes', '')}")

Next: read a few raw extractions in `data/extractions/` against the source text
before scaling past ~100 documents.